In [75]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans

In [94]:
df = pd.read_parquet(
    "../data/interim/03_administrative_district.parquet", engine="pyarrow"
)

In [77]:
df["year"].value_counts(normalize=True).sort_index() * 100

year
2018    19.382882
2019    13.269366
2020    13.599141
2021    11.809130
2022     9.845550
2023    12.287838
2024    10.683518
2025     7.862707
2026     1.259868
Name: proportion, dtype: float64

In [78]:
pd.crosstab(df["year"], df["market_type"], normalize="index") * 100

market_type,primary,secondary
year,,
2018,44.384085,55.615915
2019,64.597859,35.402141
2020,63.648102,36.351898
2021,75.680519,24.319481
2022,73.265363,26.734637
2023,73.507487,26.492513
2024,68.403138,31.596862
2025,46.836667,53.163333
2026,0.000000,100.000000


In [79]:
8 * 12 + 3

99

In [80]:
99 / 100 * 15

14.85

In [81]:
df[df["year"] == 2026]["month"].unique()

array([3, 1, 2], dtype=uint8)

In [ ]:
df_clustering = df.drop(columns=["price", "price_per_square_meter"]).sort_values("date")

In [ ]:
df_train = df_clustering[df_clustering["year"] < 2024]
df_valid = df_clustering[
    (df_clustering["year"] >= 2024) & (df_clustering["year"] < 2025)
]
df_test = df_clustering[df_clustering["year"] >= 2025]

In [ ]:
total = len(df)

train_pct = len(df_train) / total * 100
valid_pct = len(df_valid) / total * 100
test_pct = len(df_test) / total * 100

train_pct, valid_pct, test_pct

(80.19390745070083, 10.683518194303208, 9.122574354995963)

In [ ]:
import pandas as pd

result = (
    pd.DataFrame(
        {
            "train": df_train["market_type"].value_counts(normalize=True),
            "valid": df_valid["market_type"].value_counts(normalize=True),
            "test": df_test["market_type"].value_counts(normalize=True),
        }
    )
    * 100
)

result

,train,valid,test
market_type,,,
primary,63.612446,68.403138,40.368317
secondary,36.387554,31.596862,59.631683


In [86]:
df_train.columns

Index(['address', 'longitude', 'latitude', 'area', 'room_count', 'floor',
       'market_type', 'build_year', 'balcony', 'date', 'year', 'month', 'day',
       'administrative_district'],
      dtype='str')

In [ ]:
df_train = df_train.drop(columns=["market_type"])
df_valid = df_valid.drop(columns=["market_type"])
df_test = df_test.drop(columns=["market_type"])

In [ ]:
num_cols = df_train.select_dtypes(include="number").columns
cat_cols = df_train.select_dtypes(include="category").columns

In [89]:
print(num_cols)
print(cat_cols)

Index(['longitude', 'latitude', 'area', 'room_count', 'floor', 'build_year',
       'year', 'month', 'day'],
      dtype='str')
Index(['administrative_district'], dtype='str')


In [ ]:
preprocess = ColumnTransformer(
    [("num", StandardScaler(), num_cols), ("cat", OneHotEncoder(), cat_cols)]
)

In [91]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init="auto")

In [ ]:
model = Pipeline([("prep", preprocess), ("kmeans", kmeans)])

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_train = model.named_steps["prep"].fit_transform(df_train)
X_valid = model.named_steps["prep"].transform(df_valid)
X_test = model.named_steps["prep"].transform(df_test)

results = []

for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")

    train_labels = km.fit_predict(X_train)
    valid_labels = km.predict(X_valid)
    test_labels = km.predict(X_test)

    train_score = silhouette_score(X_train, train_labels)
    valid_score = silhouette_score(X_valid, valid_labels)
    test_score = silhouette_score(X_test, test_labels)

    gap_valid = train_score - valid_score
    gap_test = train_score - test_score

    results.append((k, train_score, valid_score, test_score, gap_valid, gap_test))

    print(
        f"k={k} | "
        f"train={train_score:.4f} | "
        f"valid={valid_score:.4f} | "
        f"test={test_score:.4f} | "
        f"gap(v)={gap_valid:.4f} | gap(t)={gap_test:.4f}"
    )

KeyboardInterrupt: 